# 04 - Publier avec pygeoapi

**Objectif**
- Publier sans base de données : la sortie JSON-FG d'ili2ogc est convertie en GeoJSON et servie par pygeoapi (OGC API - Features), interrogée avec `requests`, affichée avec folium.
- Publier avec une base de données : démarrer la pile Docker du dépôt (PostGIS, chargeur, pygeoapi) et comparer.

Jeu de données : les distances par rapport à la forêt, comme au notebook 03.

**Prérequis** : la première partie ne demande que Python ; la seconde (étape 9) demande Docker (avec le plugin `docker compose`) et le port 5000 libre. Elle ne tourne pas sur Binder.

## Deux voies de publication

```text
voie sans base de données (étapes 1 à 8)
  .xtf --convert-jsonfg--> JSON-FG --(reprojection)--> GeoJSON --> pygeoapi (provider GeoJSON)

voie PostGIS (étape 9, pile Docker du dépôt)
  .ili --convert-sql--> schéma + vues ; .xtf --import-xtf--> INSERT ; PostGIS --> pygeoapi (provider PostgreSQL)
```

La première convient pour essayer, pour un petit jeu de données ou pour un service sans infrastructure. Elle demande Python seulement.

## Étape 1 - Installer pygeoapi et les outils de la carte

Le serveur de développement de pygeoapi est lancé depuis ce notebook ; il écoute sur la machine locale seulement.

In [ ]:
%pip install -q pygeoapi folium requests pyproj
!mkdir -p modeles sorties

## Étape 2 - Produire le JSON-FG

Les modèles sont téléchargés (comme au notebook 03), puis `convert-jsonfg` écrit les objets du transfert, dont ceux de la VIEW `linien`.

In [ ]:
!interlis fetch-models ../data/source-xtf/ch_np_wal_v1_2.xtf ../collections/waldabstandslinien/views.ili --repo ../collections/waldabstandslinien -o modeles
!interlis convert-jsonfg ../data/source-xtf/ch_np_wal_v1_2.xtf --model ../collections/waldabstandslinien/views.ili --repo modeles -o sorties/waldabstandslinien.jsonfg

In [ ]:
import json

jsonfg = json.load(open("sorties/waldabstandslinien.jsonfg"))
nb_linien = sum(f["featureType"] == "linien" for f in jsonfg["features"])
print(jsonfg["coordRefSys"], "-", len(jsonfg["features"]), "objets, dont", nb_linien, "de type linien")

## Étape 3 - Du JSON-FG au GeoJSON

Le provider GeoJSON de pygeoapi lit un fichier GeoJSON : la géométrie est dans `geometry`, en longitude/latitude (WGS84, `OGC:CRS84`). Le JSON-FG d'ili2ogc place la géométrie dans `place`, en coordonnées suisses (EPSG:2056, est/nord en mètres). La conversion demande donc de choisir les objets d'une VIEW, de reprojeter les coordonnées et de déplacer `place` vers `geometry`.

### À vous : convertir le JSON-FG en GeoJSON

Compléter `jsonfg_vers_geojson(jsonfg, type_objet)` : elle renvoie une `FeatureCollection` GeoJSON des objets dont le `featureType` vaut `type_objet`. Chaque objet garde son `id` et ses `properties` ; sa `geometry` est le `place` d'origine, coordonnées reprojetées avec `vers_wgs84.transform(est, nord)`, qui renvoie `(longitude, latitude)`.

Indice : les coordonnées d'une `LineString` sont une liste de positions ; une position est `[est, nord]`.

In [ ]:
from pyproj import Transformer

vers_wgs84 = Transformer.from_crs(2056, 4326, always_xy=True)


def jsonfg_vers_geojson(jsonfg, type_objet):
    """Objets de `type_objet` en GeoJSON WGS84."""
    raise NotImplementedError("à compléter")

**Vérification** : la cellule suivante dit si l'exercice est réussi.

In [ ]:
geojson = jsonfg_vers_geojson(jsonfg, "linien")
assert geojson["type"] == "FeatureCollection"
assert len(geojson["features"]) == 5, f"5 objets attendus, {len(geojson['features'])} obtenus"
premier = geojson["features"][0]
assert premier["geometry"]["type"] == "LineString"
longitude, latitude = premier["geometry"]["coordinates"][0]
assert 8.5 < longitude < 9.5 and 46.8 < latitude < 47.4, f"({longitude}, {latitude}) n'est pas dans le canton de Glarus"
assert premier["id"] and "rechtsstatus" in premier["properties"]
json.dump(geojson, open("sorties/waldabstandslinien.geojson", "w"))
print("Exercice réussi :", len(geojson["features"]), "objets écrits dans sorties/waldabstandslinien.geojson")

#### Solution

À ouvrir après avoir essayé. Exécuter la cellule remplace la réponse de l'exercice ; relancer ensuite la vérification.

In [ ]:
from pyproj import Transformer

vers_wgs84 = Transformer.from_crs(2056, 4326, always_xy=True)


def jsonfg_vers_geojson(jsonfg, type_objet):
    """Objets de `type_objet` en GeoJSON WGS84."""

    def reprojeter(coordonnees):
        if isinstance(coordonnees[0], (int, float)):
            return list(vers_wgs84.transform(*coordonnees[:2]))
        return [reprojeter(c) for c in coordonnees]

    objets = []
    for objet in jsonfg["features"]:
        if objet["featureType"] != type_objet:
            continue
        geometrie = {"type": objet["place"]["type"], "coordinates": reprojeter(objet["place"]["coordinates"])}
        objets.append({"type": "Feature", "id": objet["id"], "geometry": geometrie, "properties": objet["properties"]})
    return {"type": "FeatureCollection", "features": objets}

## Étape 4 - Écrire la configuration de pygeoapi

pygeoapi se configure par un fichier YAML (`server`, `metadata`, `resources`). Une ressource de type `collection` a un ou plusieurs providers ; ici un provider `feature` de nom `GeoJSON`, qui lit le fichier de l'étape 3. `id_field` désigne l'identifiant des objets.

Le serveur écoute sur le port 5001 (le port 5000 est celui de la pile Docker de l'étape 9) ; en cas de conflit, changer `port` et `url`.

In [ ]:
%%writefile sorties/pygeoapi-config.yml
server:
    bind:
        host: 127.0.0.1
        port: 5001
    url: http://127.0.0.1:5001
    mimetype: application/json; charset=UTF-8
    encoding: utf-8
    languages:
        - fr-CH
    limits:
        default_items: 20
        max_items: 100
logging:
    level: WARNING
metadata:
    identification:
        title: Distances par rapport à la forêt (exercice)
        description: Jeu de données du dépôt, converti par ili2ogc et servi par pygeoapi.
        keywords:
            - interlis
        terms_of_service: https://opendata.swiss/en/terms-of-use/#terms_by
        url: https://github.com/geostandards-ch/ili2ogc-pygeoapi-demo
    license:
        name: MIT
        url: https://github.com/geostandards-ch/ili2ogc-pygeoapi-demo/blob/main/LICENSE.md
    provider:
        name: exercice
        url: https://github.com/geostandards-ch/ili2ogc-pygeoapi-demo
    contact:
        name: exercice
        email: exercice@example.org
resources:
    waldabstandslinien:
        type: collection
        title: Distances par rapport à la forêt
        description: Lignes de distance par rapport à la forêt (canton de Glarus).
        keywords:
            - forêt
        extents:
            spatial:
                bbox: [9.0, 47.0, 9.3, 47.2]
                crs: http://www.opengis.net/def/crs/OGC/1.3/CRS84
        providers:
            - type: feature
              name: GeoJSON
              data: sorties/waldabstandslinien.geojson
              id_field: id

In [ ]:
%env PYGEOAPI_CONFIG=sorties/pygeoapi-config.yml
%env PYGEOAPI_OPENAPI=sorties/pygeoapi-openapi.yml
!pygeoapi openapi generate sorties/pygeoapi-config.yml --output-file sorties/pygeoapi-openapi.yml

`pygeoapi openapi generate` a écrit le document OpenAPI de l'instance : pygeoapi le lit au démarrage avec la configuration (variables d'environnement `PYGEOAPI_CONFIG` et `PYGEOAPI_OPENAPI`, fixées par `%env`). Le chemin `data` de la configuration est relatif au dossier du notebook, d'où le serveur est lancé.

## Étape 5 - Démarrer le serveur en arrière-plan

`%%bash --bg --proc serveur` lance le script en arrière-plan et garde le processus dans la variable `serveur` ; la sortie du serveur va dans un fichier journal. Le noyau termine ce processus à sa fermeture.

In [ ]:
%%bash --bg --proc serveur
exec pygeoapi serve > sorties/pygeoapi.log 2>&1

In [ ]:
%%bash
for i in $(seq 60); do
  curl -sf -o /dev/null http://127.0.0.1:5001/ && echo "pygeoapi répond sur http://127.0.0.1:5001" && exit 0
  sleep 0.5
done
echo "pygeoapi n'a pas répondu" ; tail -5 sorties/pygeoapi.log ; exit 1

## Étape 6 - Interroger l'API

OGC API - Features expose la page d'accueil (`/`), la liste des collections (`/collections`), une collection et ses objets (`/collections/{id}/items`). Le paramètre `f=json` demande du JSON.

In [ ]:
import requests

URL = "http://127.0.0.1:5001"
accueil = requests.get(URL, params={"f": "json"}).json()
print("Titre :", accueil["title"])
collections = requests.get(f"{URL}/collections", params={"f": "json"}).json()
print("Collections :", [c["id"] for c in collections["collections"]])

objets = requests.get(f"{URL}/collections/waldabstandslinien/items", params={"f": "json", "limit": 2}).json()
print("numberMatched :", objets["numberMatched"], "- numberReturned :", objets["numberReturned"])
print("Propriétés du premier objet :", objets["features"][0]["properties"])

Les propriétés comprennent les attributs suivis par référence (`typ_code`, `typ_bezeichnung`, `typ_verbindlichkeit`) : ils viennent de `convert-jsonfg`, sans base de données.

### À vous : filtrer par emprise

Stocker dans `params` les paramètres de requête qui ne gardent que les objets situés dans l'emprise ouest 9.05, sud 47.09, est 9.10, nord 47.12 (paramètre `bbox`, au format `ouest,sud,est,nord`), au plus 10 objets (`limit`).

In [ ]:
params = {}

**Vérification** : la cellule suivante dit si l'exercice est réussi.

In [ ]:
reponse = requests.get(f"{URL}/collections/waldabstandslinien/items", params={**params, "f": "json"})
assert reponse.status_code == 200, reponse.text[:300]
identifiants = sorted(f["id"] for f in reponse.json()["features"])
assert identifiants == ["186", "188"], f"objets attendus : 186 et 188 ; obtenus : {identifiants}"
print("Exercice réussi : objets dans l'emprise :", identifiants)

#### Solution

À ouvrir après avoir essayé. Exécuter la cellule remplace la réponse de l'exercice ; relancer ensuite la vérification.

In [ ]:
params = {"bbox": "9.05,47.09,9.10,47.12", "limit": 10}

## Étape 7 - Afficher la carte

Les objets sont lus dans l'API (et non dans le fichier), puis dessinés avec folium. La couleur du trait vient du SLD produit par `convert-sld` à partir de la représentation du dépôt (notebook 02) : la même définition graphique alimente ici la carte du notebook. Le fond de carte OpenStreetMap est chargé par le navigateur.

In [ ]:
!interlis fetch-models ../collections/waldabstandslinien/symbology.ili --repo ../collections/waldabstandslinien -o modeles
!interlis convert-sld ../collections/waldabstandslinien/symbology.ili --repo modeles --repo ../collections/waldabstandslinien \
    --sign-xtf ../collections/waldabstandslinien/signs.xtf --graphic Waldabstand_Graphics -o sorties/waldabstand.sld
!grep stroke sorties/waldabstand.sld

In [ ]:
import re

import folium

couleur = re.search(r'name="stroke">(#\w+)', open("sorties/waldabstand.sld").read()).group(1)
donnees_api = requests.get(f"{URL}/collections/waldabstandslinien/items", params={"f": "json", "limit": 100}).json()
carte = folium.Map(location=[47.10, 9.12], zoom_start=11, tiles="OpenStreetMap")
folium.GeoJson(
    donnees_api,
    name="Distances par rapport à la forêt",
    style_function=lambda objet: {"color": couleur, "weight": 4},
    tooltip=folium.GeoJsonTooltip(fields=["rechtsstatus", "publiziertab"]),
).add_to(carte)
carte.fit_bounds(carte.get_bounds())
carte.save("sorties/carte.html")
assert len(donnees_api["features"]) == 5
carte

## Étape 8 - Arrêter le serveur

Toujours terminer le processus fils, y compris si une cellule précédente a échoué.

In [ ]:
serveur.terminate()
try:
    requests.get(URL, timeout=2)
    raise AssertionError("le serveur répond encore")
except requests.ConnectionError:
    print("Serveur arrêté, port 5001 libéré.")

## Limites de la voie sans base de données

- **Catalogues.** Les attributs obtenus par référence (`typ_code`, `typ_bezeichnung`...) sont écrits par `convert-jsonfg` et servis ici. Si la référence pointe vers un catalogue absent du transfert, il faut le passer avec `--catalog` (notebook 03) ; sinon l'attribut manque et la commande le signale (`JSONFG-VIEW-PATH-UNRESOLVED`, code de sortie 2).
- **Pas de style côté serveur.** Le provider GeoJSON sert des objets, pas des cartes : le style SLD n'est ici qu'une source de couleur pour folium. OGC API - Maps demande un moteur de rendu (MapServer) et une source géographique compatible.
- **Un seul système de coordonnées.** Le GeoJSON est en WGS84 ; la configuration du dépôt propose aussi EPSG:2056.

## Étape 9 - La voie PostGIS : la pile Docker du dépôt

La pile du dépôt (`docker-compose.yml`, à la racine) publie plusieurs jeux de données avec ces compléments. Trois services :

1. **postgis** : la base ;
2. **loader** (`loader/load.py`) : pour chaque dossier de `collections/`, lance `fetch-models`, `validate-xtf`, `convert-sql` et `import-xtf`, exécute les deux scripts SQL dans un schéma dédié, puis lance `convert-sld` pour chaque `GRAPHIC` ;
3. **pygeoapi** (`pygeoapi/pygeoapi-config.yml`) : provider `PostgreSQL` sur la vue `linien` (colonnes `geometrie`, clé `t_ili_tid`), `storage_crs` EPSG:2056, et provider `map` (MapScript) qui dessine avec le SLD généré.

Le démarrage construit les images et charge les données : compter quelques minutes la première fois, et un accès à Internet (le chargeur appelle `fetch-models`).

La seconde commande `up -d` relance le chargeur s'il a échoué au tout premier démarrage, quand la base annonce être prête mais n'accepte pas encore de connexions ; sans effet sinon.

In [ ]:
!cd .. && docker compose --ansi never up -d --build 2>&1 | tail -5
!cd .. && docker compose --ansi never up -d 2>&1 | tail -3

In [ ]:
%%bash
for i in $(seq 120); do
  curl -sf "http://localhost:5000/collections/waldabstandslinien/items?f=json&limit=1" | grep -q typ_bezeichnung && echo "pygeoapi (PostGIS) répond sur http://localhost:5000" && exit 0
  sleep 1
done
echo "pas de données : voir le journal du chargeur ci-dessous (le premier démarrage peut échouer si la base n'accepte pas encore les connexions ; relancer alors la cellule précédente)" ; cd .. && docker compose logs --tail 15 loader ; exit 1

In [ ]:
postgis = "http://localhost:5000"
print("Collections :", [c["id"] for c in requests.get(f"{postgis}/collections", params={"f": "json"}).json()["collections"]])
objet = requests.get(f"{postgis}/collections/waldabstandslinien/items", params={"f": "json", "limit": 1}).json()["features"][0]
print("Propriétés du premier objet :", objet["properties"])
assert "typ_bezeichnung" in objet["properties"], "les attributs obtenus par référence devraient être présents"
print("Propriétés absentes de la voie sans base :", sorted(set(objet["properties"]) - set(objets["features"][0]["properties"])))
print("Propriétés absentes de la voie PostGIS :", sorted(set(objets["features"][0]["properties"]) - set(objet["properties"])))

Les deux voies fournissent les attributs suivis par référence (`typ_code`, `typ_bezeichnung`...). Elles diffèrent ici par les colonnes de la vue SQL : `t_ili_tid`, et les attributs sans valeur dans ce transfert (`publiziertbis`, `bemerkungen`, `typ_abkuerzung`), que PostgreSQL renvoie à `null` et que le JSON-FG n'écrit pas. La carte, elle, est dessinée côté serveur avec le SLD généré (OGC API - Maps) :

In [ ]:
!curl -s -o sorties/carte_serveur.png "http://localhost:5000/collections/waldabstandslinien/map?f=png"
from IPython.display import Image
Image("sorties/carte_serveur.png")

| | Sans base (étapes 1 à 8) | PostGIS (pile Docker) |
|---|---|---|
| Entrée de pygeoapi | GeoJSON issu de `convert-jsonfg` | vue SQL issue de `convert-sql` et `import-xtf` |
| Provider | `GeoJSON` | `PostgreSQL`, et `MapScript` pour les cartes |
| Attributs par référence | présents (catalogues : `--catalog`) | présents |
| Attributs sans valeur | non écrits | colonnes à `null` |
| Cartes OGC API - Maps, styles SLD | non | oui |
| Prérequis | Python | Docker |

Arrêter la pile quand on a fini (les conteneurs tournent tant qu'on ne les arrête pas) :

In [ ]:
!cd .. && docker compose --ansi never down 2>&1 | tail -8

Pour ajouter un jeu de données à la pile : créer `collections/<nom>/` avec `views.ili`, `symbology.ili`, `signs.xtf`, déclarer ses transferts dans `TRANSFERS` (`loader/load.py`) et sa ressource dans `pygeoapi/pygeoapi-config.yml`.